In [3]:
#Extract
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
from sqlalchemy.engine import URL

customers = pd.read_excel(r'C:\Users\hp\Downloads\customers.xlsx')
products = pd.read_excel(r'C:\Users\hp\Downloads\products.xlsx')
sales = pd.read_excel(r'C:\Users\hp\Downloads\Sales.xlsx')

print("customers")
display(customers.head())

print("products")
display(products.head())

print("sales")
display(sales.head())

customers


,CustomerID,CustomerName,Gender,City,Country,Phone,RegistrationDate
0,CUST0001,Fatima Hassan,Male,Mogadishu,Somalia,2.526327e+11,2024-07-17
1,CUST0002,Asha Ali,Male,Hargeisa,Somalia,NaN,2025-05-14
2,CUST0003,Sahra Ahmed,Male,Berbera,Somalia,2.526380e+11,2023-02-24
3,CUST0004,Sahra Warsame,Male,Burao,Somalia,2.526381e+11,2024-07-23
4,CUST0005,Hassan Mohamed,Male,Burao,Somalia,2.526326e+11,2023-11-15


products


,ProductID,ProductName,Category,UnitCost,UnitPrice,Supplier,Active
0,PROD001,Wireless Mouse,Electronics,101.38,144.24,Supplier B,Yes
1,PROD002,Keyboard,Electronics,45.72,62.68,Supplier B,No
2,PROD003,USB Flash Drive,Electronics,117.32,165.50,Supplier A,Yes
3,PROD004,Headphones,electronics,55.08,75.46,Supplier B,Yes
4,PROD005,Power Bank,Electronics,120.52,163.12,Supplier C,No


sales


,OrderID,OrderDate,CustomerID,ProductID,Quantity,Discount,PaymentMethod,SalesChannel
0,ORD000001,2024-10-16,CUST0037,PROD023,12,0.10,Cash,Wholesale
1,ORD000002,2026-03-17,CUST0030,PROD009,11,0.15,Card,Online
2,ORD000003,2024-08-18,CUST0015,PROD018,11,0.00,Mobile Money,Store
3,ORD000004,2024-06-18,CUST0010,PROD002,5,0.15,Mobile Money,Online
4,ORD000005,2025-12-06,CUST0016,PROD015,5,0.20,Bank Transfer,Online


In [4]:
#transform
# =========================
# CLEAN CUSTOMERS
# =========================

customers = customers.drop_duplicates(
    subset=["CustomerID"]
)

customers = customers.dropna(
    subset=["CustomerID", "CustomerName"]
)

customers["CustomerName"] = (
    customers["CustomerName"].str.strip().str.title()
)

customers["City"] = (
    customers["City"].str.strip().str.title()
)


# =========================
# CLEAN PRODUCTS
# =========================

products = products.drop_duplicates(
    subset=["ProductID"]
)

products = products.dropna(
    subset=["ProductID", "ProductName"]
)

products["ProductName"] = (
    products["ProductName"].str.strip().str.title()
)

products["Category"] = (
    products["Category"].str.strip().str.title()
)


# =========================
# CLEAN SALES
# =========================

sales = sales.drop_duplicates(
    subset=["OrderID"]
)

sales = sales.dropna(
    subset=["OrderID", "CustomerID", "ProductID", "OrderDate"]
)

sales["OrderDate"] = pd.to_datetime(
    sales["OrderDate"], errors="coerce"
)

sales["Quantity"] = pd.to_numeric(
    sales["Quantity"], errors="coerce"
)

sales["Discount"] = pd.to_numeric(
    sales["Discount"], errors="coerce"
)

sales["Discount"] = sales["Discount"].fillna(0)

sales = sales.dropna(
    subset=["OrderDate", "Quantity"]
)

sales = sales[sales["Quantity"] > 0]

sales = sales[sales["Discount"].between(0, 1)]


print("Data Cleaning Completed!")

print("Customers:", len(customers))
print("Products:", len(products))
print("Sales:", len(sales))

Data Cleaning Completed!
Customers: 100
Products: 24
Sales: 600


In [5]:
# data merge
# Merge Sales with Products

sales = pd.merge(
    sales,
    products[["ProductID", "UnitPrice", "UnitCost"]],
    on="ProductID",
    how="inner"
)


# Merge Sales with Customers

sales = pd.merge(
    sales,
    customers[["CustomerID", "CustomerName"]],
    on="CustomerID",
    how="inner"
)


# Calculate Revenue

sales["GrossRevenue"] = (
    sales["Quantity"] * sales["UnitPrice"]
)

# Calculate Discount

sales["DiscountAmount"] = (
    sales["GrossRevenue"] * sales["Discount"]
)

# Calculate Net Revenue

sales["NetRevenue"] = (
    sales["GrossRevenue"] - sales["DiscountAmount"]
)

# Calculate Total Cost

sales["TotalCost"] = (
    sales["Quantity"] * sales["UnitCost"]
)

# Calculate Profit

sales["Profit"] = (
    sales["NetRevenue"] - sales["TotalCost"]
)


display(sales.head())

print("Transformation Completed!")

,OrderID,OrderDate,CustomerID,ProductID,Quantity,Discount,PaymentMethod,SalesChannel,UnitPrice,UnitCost,CustomerName,GrossRevenue,DiscountAmount,NetRevenue,TotalCost,Profit
0,ORD000001,2024-10-16,CUST0037,PROD023,12,0.10,Cash,Wholesale,126.23,100.70,Maryan Farah,1514.76,151.4760,1363.2840,1208.40,154.8840
1,ORD000002,2026-03-17,CUST0030,PROD009,11,0.15,Card,Online,58.13,41.30,Khadija Omar,639.43,95.9145,543.5155,454.30,89.2155
2,ORD000003,2024-08-18,CUST0015,PROD018,11,0.00,Mobile Money,Store,221.09,178.92,Nimco Muse,2431.99,0.0000,2431.9900,1968.12,463.8700
3,ORD000004,2024-06-18,CUST0010,PROD002,5,0.15,Mobile Money,Online,62.68,45.72,Hodan Ismail,313.40,47.0100,266.3900,228.60,37.7900
4,ORD000005,2025-12-06,CUST0016,PROD015,5,0.20,Bank Transfer,Online,42.90,35.63,Abdirahman Warsame,214.50,42.9000,171.6000,178.15,-6.5500


Transformation Completed!


In [6]:
# Diyaari Star Schema
# =========================
# DIM CUSTOMER
# =========================

dim_customer = customers.copy()

dim_customer.insert(
    0,
    "CustomerKey",
    range(1, len(dim_customer) + 1)
)


# =========================
# DIM PRODUCT
# =========================

dim_product = products.copy()

dim_product.insert(
    0,
    "ProductKey",
    range(1, len(dim_product) + 1)
)


# =========================
# DIM DATE
# =========================

dim_date = pd.DataFrame({
    "FullDate": pd.to_datetime(
        sales["OrderDate"].dt.date.unique()
    )
})

dim_date = dim_date.sort_values("FullDate").reset_index(drop=True)

dim_date.insert(
    0,
    "DateKey",
    dim_date["FullDate"].dt.strftime("%Y%m%d").astype(int)
)

dim_date["Day"] = dim_date["FullDate"].dt.day
dim_date["Month"] = dim_date["FullDate"].dt.month
dim_date["MonthName"] = dim_date["FullDate"].dt.month_name()
dim_date["Quarter"] = dim_date["FullDate"].dt.quarter
dim_date["Year"] = dim_date["FullDate"].dt.year


# =========================
# FACT SALES
# =========================

fact_sales = sales.merge(
    dim_customer[["CustomerID", "CustomerKey"]],
    on="CustomerID",
    how="inner"
)

fact_sales = fact_sales.merge(
    dim_product[["ProductID", "ProductKey"]],
    on="ProductID",
    how="inner"
)

fact_sales["FullDate"] = fact_sales["OrderDate"].dt.normalize()

fact_sales = fact_sales.merge(
    dim_date[["DateKey", "FullDate"]],
    on="FullDate",
    how="inner"
)


# Select Fact Table Columns

fact_sales = fact_sales[
    [
        "OrderID",
        "CustomerKey",
        "ProductKey",
        "DateKey",
        "Quantity",
        "UnitPrice",
        "Discount",
        "GrossRevenue",
        "DiscountAmount",
        "NetRevenue",
        "TotalCost",
        "Profit",
        "PaymentMethod",
        "SalesChannel"
    ]
]


print("Star Schema Created!")

display(fact_sales.head())

Star Schema Created!


,OrderID,CustomerKey,ProductKey,DateKey,Quantity,UnitPrice,Discount,GrossRevenue,DiscountAmount,NetRevenue,TotalCost,Profit,PaymentMethod,SalesChannel
0,ORD000001,37,23,20241016,12,126.23,0.10,1514.76,151.4760,1363.2840,1208.40,154.8840,Cash,Wholesale
1,ORD000002,30,9,20260317,11,58.13,0.15,639.43,95.9145,543.5155,454.30,89.2155,Card,Online
2,ORD000003,15,18,20240818,11,221.09,0.00,2431.99,0.0000,2431.9900,1968.12,463.8700,Mobile Money,Store
3,ORD000004,10,2,20240618,5,62.68,0.15,313.40,47.0100,266.3900,228.60,37.7900,Mobile Money,Online
4,ORD000005,16,15,20251206,5,42.90,0.20,214.50,42.9000,171.6000,178.15,-6.5500,Bank Transfer,Online


In [14]:
import pyodbc
from sqlalchemy import create_engine
from urllib.parse import quote_plus

server = r'DESKTOP-RK05MDR\SAMATAR1234'
database = 'RetailDataWarehouse'

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    + quote_plus(connection_string)
)

engine = create_engine(connection_url)

# Test connection
with engine.connect() as conn:
    print("SQL SERVER CONNECTED SUCCESSFULLY!")

SQL SERVER CONNECTED SUCCESSFULLY!


In [15]:
# LOAD DIMENSION TABLES

dim_customer.to_sql(
    name="DimCustomer",
    con=engine,
    if_exists="replace",
    index=False
)

dim_product.to_sql(
    name="DimProduct",
    con=engine,
    if_exists="replace",
    index=False
)

dim_date.to_sql(
    name="DimDate",
    con=engine,
    if_exists="replace",
    index=False
)

# LOAD FACT TABLE

fact_sales.to_sql(
    name="FactSales",
    con=engine,
    if_exists="replace",
    index=False
)

print("All Tables Successfully Loaded!")

All Tables Successfully Loaded!
